In [21]:
import numpy as np 
import pandas as pd 

from sklearn.model_selection import train_test_split
from sklearn.feature_extraction.text import CountVectorizer

from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, f1_score, precision_score, recall_score

import re
import string

from nltk.corpus import stopwords
from nltk.stem import WordNetLemmatizer

import mlflow

In [22]:
df = pd.read_csv('https://raw.githubusercontent.com/campusx-official/jupyter-masterclass/main/tweet_emotions.csv').drop(columns=['tweet_id'])
df.head()

,sentiment,content
0,empty,@tiffanylue i know i was listenin to bad habi...
1,sadness,Layin n bed with a headache ughhhh...waitin o...
2,sadness,Funeral ceremony...gloomy friday...
3,enthusiasm,wants to hang out with friends SOON!
4,neutral,@dannycastillo We want to trade with someone w...


In [23]:
import nltk 
nltk.download('stopwords')
nltk.download('wordnet')

[nltk_data] Downloading package stopwords to /home/anni/nltk_data...
[nltk_data]   Package stopwords is already up-to-date!
[nltk_data] Downloading package wordnet to /home/anni/nltk_data...
[nltk_data]   Package wordnet is already up-to-date!


True

In [24]:
lemmatizer = WordNetLemmatizer()
stop_words = set(stopwords.words('english'))


def lemmatization(text):
    text = text.split()
    text = [lemmatizer.lemmatize(word) for word in text]
    return ' '.join(text)

def remove_stop_words(text):
    text = [word for word in str(text).split() if word not in stop_words]
    return ' '.join(text)

def remove_numbers(text):
    text = ''.join(char for char in text if not char.isdigit())
    return text

def lowercase(text):
    text = text.split()
    text = [word.lower() for word in text]
    return ' '.join(text)

def remove_punctuations(text):
    text = re.sub('[%s]' % re.escape(string.punctuation), ' ', text)
    text = text.replace(':', '')
    text = re.sub(r'\s+', ' ', text).strip()
    return text

def remove_urls(text):
    url_pattern = re.compile(r'https?://\S+|www\.\S+')
    return url_pattern.sub(r'', text)

def normalize_text(df):
    try:
        df['content'] = df['content'].apply(lowercase)
        df['content'] = df['content'].apply(remove_urls)
        df['content'] = df['content'].apply(remove_punctuations)
        df['content'] = df['content'].apply(remove_numbers)
        df['content'] = df['content'].apply(remove_stop_words)
        df['content'] = df['content'].apply(lemmatization)
        return df
    except Exception as e:
        print(f'error in text_normalization: {e}')
        raise 

In [25]:
df = normalize_text(df)

In [26]:
x = df['sentiment'].isin(['happiness', 'sadness'])
df = df[x]

In [27]:
df['sentiment'] = df['sentiment'].replace({'sadness': 0, 'happiness': 1})
df.head()

/tmp/ipykernel_62683/2161997397.py:1: FutureWarning: Downcasting behavior in `replace` is deprecated and will be removed in a future version. To retain the old behavior, explicitly call `result.infer_objects(copy=False)`. To opt-in to the future behavior, set `pd.set_option('future.no_silent_downcasting', True)`
  df['sentiment'] = df['sentiment'].replace({'sadness': 0, 'happiness': 1})


,sentiment,content
1,0,layin n bed headache ughhhh waitin call
2,0,funeral ceremony gloomy friday
6,0,sleep im thinking old friend want married damn...
8,0,charviray charlene love miss
9,0,kelcouch sorry least friday


In [28]:
vectorizer = CountVectorizer(max_features=2000)

x = vectorizer.fit_transform(df['content'])
y = df['sentiment']

In [29]:
x_train, x_test, y_train, y_test = train_test_split(x, y, test_size=.2, random_state=42)

In [20]:
import dagshub
dagshub.init(repo_owner='anni0955', repo_name='mlops-mini-project', mlflow=True)

mlflow.set_tracking_uri('https://dagshub.com/anni0955/mlops-mini-project.mlflow')

mlflow.set_experiment('Logistic Regression Baseline')

Accessing as anni0955

Initialized MLflow to track repo "anni0955/mlops-mini-project"

Repository anni0955/mlops-mini-project initialized!

2026/03/28 11:50:17 INFO mlflow.tracking.fluent: Experiment with name 'Logistic Regression Baseline' does not exist. Creating a new experiment.


<Experiment: artifact_location='mlflow-artifacts:/b54fdf8d46234d4ebe8641c171c40902', creation_time=1774678821680, experiment_id='0', last_update_time=1774678821680, lifecycle_stage='active', name='Logistic Regression Baseline', tags={}, workspace='default'>

In [33]:
with mlflow.start_run():
    mlflow.log_param('vectorize', 'Bag of Words')
    mlflow.log_param('num_features', 2000)
    mlflow.log_param('test_size', .2)

    model = LogisticRegression()
    model.fit(x_train, y_train)

    mlflow.log_param('model', 'Logistic Regression')

    y_pred = model.predict(x_test)

    accuracy = accuracy_score(y_test, y_pred)
    f1 = f1_score(y_test, y_pred)
    precision = precision_score(y_test, y_pred)
    recall = recall_score(y_test, y_pred)

    mlflow.log_metric('accuracy', accuracy)
    mlflow.log_metric('f1', f1)
    mlflow.log_metric('precision', precision)
    mlflow.log_metric('recall', recall)

    mlflow.sklearn.log_model(model, 'model')

    import os 
    notebook_path = 'exp1_baseline_model.ipynb'
    os.system(f'jupyter nbconvert --to notebook --execute --inplace {notebook_path}')

    mlflow.log_artifact(str(notebook_path))


    print(f'accuracy: {accuracy}')
    print(f'f1: {f1}')
    print(f'precision: {precision}')
    print(f'recall: {recall}')


2026/03/28 14:33:51 WARNING mlflow.models.model: `artifact_path` is deprecated. Please use `name` instead.
2026/03/28 14:33:56 WARNING mlflow.sklearn: Saving scikit-learn models in the pickle or cloudpickle format requires exercising caution because these formats rely on Python's object serialization mechanism, which can execute arbitrary code during deserialization. The recommended safe alternative is the 'skops' format. For more information, see: https://scikit-learn.org/stable/model_persistence.html
usage: jupyter [-h] [--version] [--config-dir] [--data-dir] [--runtime-dir]
               [--paths] [--json] [--debug]
               [subcommand]

Jupyter: Interactive Computing

positional arguments:
  subcommand     the subcommand to launch

options:
  -h, --help     show this help message and exit
  --version      show the versions of core jupyter packages and exit
  --config-dir   show Jupyter config dir
  --data-dir     show Jupyter data dir
  --runtime-dir  show Jupyter runtime d

accuracy: 0.7812048192771084
f1: 0.7778864970645792
precision: 0.7725947521865889
recall: 0.7832512315270936
🏃 View run delightful-snail-712 at: https://dagshub.com/anni0955/mlops-mini-project.mlflow/#/experiments/0/runs/fd748c6bf17f4e839e832c331f7acd91
🧪 View experiment at: https://dagshub.com/anni0955/mlops-mini-project.mlflow/#/experiments/0
